In [ ]:
import pandas as pd
from wordcloud import WordCloud
import matplotlib.pyplot as plt
from collections import Counter
import re
import random
import matplotlib
from PIL import Image
import numpy as np

matplotlib.use('Agg')

# ===============================
# 1. 데이터 로드
# ===============================
df = pd.read_csv('./data/bx_crawling_results_부양.csv')
full_text = " ".join(df['title'].astype(str)) + " " + " ".join(df['snippet'].astype(str))

# ===============================
# 2. 한글만 추출
# ===============================
words = re.findall(r'[가-힣]{2,}', full_text)

# ===============================
# 3. 제외 패턴 (부분 포함 기준)
# ===============================
exclude_patterns = [
    '양가','결혼기념일','연말정산','부양가족','인적공제',
    '부모님자녀','자녀각각','각각','공제',
    '전국','모임','카페','블로그','커뮤니티',
    '의사','간호사','간호조무사','병원','센터','협회',
    '네이버','하우스','마켓','바로가기','저장','AI',
    '방법','절차','신청방법','관련','이유','내용','경우',
    '지극정성','효자','효녀','며느리','사위','시어머니','시아버지'
]

# ===============================
# 4. 주제 앵커 키워드 (+ 아이 관련만 추가)
# ===============================
theme_keywords = [
    '부양','간병','돌봄','병수발','치매',
    '노인','부모','가족','독거노인',
    '힘들','버겁','스트레스','짜증','분노',
    '우울','불안','지침','탈진','소진',
    '막막','한계','포기','외로움','압박',
    '독박','혼자','책임','희생','방치','고립',
    '비용','간병비','생활비','병원비','부담','생계',
    '퇴사','경력단절','불면','수면부족',
    '건강악화','일상붕괴','생활붕괴',
    '장기요양','요양','등급','탈락','지원금','방문요양',
    '노후','노후불안','미래불안',

    # 아이 / 자식
    '아이','자식','자녀','미성년','육아','양육','아이돌봄'
]

# ===============================
# 5. 빈도 계산 + 앵커 필터
# ===============================
word_counts = Counter(words)
final_counts = {}

for word, count in word_counts.items():
    if any(ex in word for ex in exclude_patterns):
        continue
    if any(theme in word or word in theme for theme in theme_keywords):
        final_counts[word] = count

# ===============================
# 6. 🍀 네잎클로버 마스크 (흰색 제외)
# ===============================
mask_img = Image.open('./data/iphone.jpg').convert('RGB')
mask_np = np.array(mask_img)

# 흰색 영역 제거 (거의 흰색이면 제외)
mask = np.where(
    (mask_np[:, :, 0] > 240) &
    (mask_np[:, :, 1] > 240) &
    (mask_np[:, :, 2] > 240),
    255,   # 배경
    0      # 클로버 영역
)

# ===============================
# 7. 컬러 (초록 + 다크그린 + 그레이)
# ===============================
clover_colors = [
    "#2E7D32",  # 다크 그린
    "#388E3C",
    "#4CAF50",
    "#66BB6A",
    "#1B5E20",
    "#2B2B2B",
    "#4A4A4A"
]

def clover_color_func(*args, **kwargs):
    return random.choice(clover_colors)

# ===============================
# 8. 워드클라우드 생성
# ===============================
wordcloud = WordCloud(
    font_path='C:/Windows/Fonts/malgun.ttf',
    background_color='white',
    width=1200,
    height=1200,
    mask=mask,
    max_words=500,
    max_font_size=150,
    min_font_size=8,
    relative_scaling=0.12,
    prefer_horizontal=1.0,
    color_func=clover_color_func,
    margin=1,
    contour_width=10,  # 2 → 0 (윤곽선 제거)
    contour_color='black'
).generate_from_frequencies(final_counts)

# ===============================
# 9. 저장
# ===============================
plt.figure(figsize=(10, 10))
plt.imshow(wordcloud, interpolation='bilinear')
plt.axis('off')
plt.tight_layout(pad=0)
plt.savefig('care_burden_clover_wordcloud.png', dpi=300, bbox_inches='tight')
plt.close()